## Data Cleaning Pipeline

### 1. Load the necessary packages

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#### Set up the path for working directory

In [2]:
# Input folder containing the combined residential datasets
INPUT_DIR = Path("output")

# Output folder for cleaned datasets
CLEANED_DIR = Path("cleaned_data")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print("Input folder:", INPUT_DIR)
print("Cleaned output folder:", CLEANED_DIR)

Input folder: output
Cleaned output folder: cleaned_data


### 2. Read the datasets

In [3]:
# (combined_sold/list_residential_with_rate.csv)

sold_raw = pd.read_csv(
    INPUT_DIR / "combined_sold_residential_with_mortgage_rates.csv",
    low_memory=False
)

list_raw = pd.read_csv(
    INPUT_DIR / "combined_list_residential_with_mortgage_rates.csv",
    low_memory=False
)

print("Sold raw shape:", sold_raw.shape)
print("List raw shape:", list_raw.shape)

Sold raw shape: (448033, 89)
List raw shape: (615739, 89)


#### Copy the original dataset

In [4]:
sold_clean = sold_raw.copy()
list_clean = list_raw.copy()

print("Sold clean shape:", sold_clean.shape)
print("List clean shape:", list_clean.shape)

Sold clean shape: (448033, 89)
List clean shape: (615739, 89)


### 3. Create the cleaning log to document any change in dataset

In [5]:
cleaning_log = []


def add_cleaning_log(
    dataset_name,
    step,
    rows_before,
    rows_after,
    cols_before,
    cols_after,
    notes=""
):
    cleaning_log.append({
        "dataset": dataset_name,
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "columns_before": cols_before,
        "columns_after": cols_after,
        "columns_removed": cols_before - cols_after,
        "notes": notes
    })

#### Purpose of Cleaning Log :
This cleaning log documents each transformation applied to the Sold and List datasets. It records before-and-after row and column counts, the number of rows or columns changed, and a brief explanation of each cleaning decision.

### 4. Initial Data Inspection

In [6]:
# Check data shape, columns, and data type
def inspect_dataset(df, dataset_name):
    print("\n" + "-" * 60)
    print(dataset_name)
    print("-" * 60)

    print("Shape:", df.shape)
    print("\nData types:")
    print(df.dtypes.value_counts())

    print("\nDuplicate full rows:")
    print(df.duplicated().sum())

    print("\nTotal missing values:")
    print(df.isna().sum().sum())


inspect_dataset(sold_clean, "Sold Dataset")
inspect_dataset(list_clean, "List Dataset")


------------------------------------------------------------
Sold Dataset
------------------------------------------------------------
Shape: (448033, 89)

Data types:
object     56
float64    30
int64       3
Name: count, dtype: int64

Duplicate full rows:
48

Total missing values:
12864595

------------------------------------------------------------
List Dataset
------------------------------------------------------------
Shape: (615739, 89)

Data types:
object     51
float64    34
int64       4
Name: count, dtype: int64

Duplicate full rows:
0

Total missing values:
18752296


In [7]:
# Check dataset time range is from 202401 to 202606
print("Sold year_month range:")
print(sold_clean["year_month"].min(), sold_clean["year_month"].max())

print("\nList year_month range:")
print(list_clean["year_month"].min(), list_clean["year_month"].max())

Sold year_month range:
2024-01 2026-06

List year_month range:
2024-01 2026-06


In [8]:
# Check each month csv file rows count in sold dataset
#print(
#    sold_clean["year_month"]
#    .value_counts(dropna=False)
#    .sort_index()
#    .to_string()
#)

In [9]:
# Check each month csv file rows count in list dataset
#print(
#    list_clean["year_month"]
#    .value_counts(dropna=False)
#    .sort_index()
#    .to_string()
#)

#### Summary of Initial Data Inspection:

The Sold dataset contained 448,033 rows and 89 columns, while the List dataset contained 615,739 rows and 89 columns. Both datasets covered the period from January 2024 through June 2026. Most columns were stored as object or float data types and required further type validation. The Sold dataset contained 48 fully duplicated rows, while no full-row duplicates were found in the List dataset. Both datasets also contained substantial missing values that required column-level review.

### 5. Standardize Column Names

In [10]:
sold_clean.columns = sold_clean.columns.str.strip()
list_clean.columns = list_clean.columns.str.strip()

print("Sold Clean Columns List")
print(sold_clean.columns.tolist())

print("List Clean Columns List")
print(list_clean.columns.tolist())

Sold Clean Columns List
['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeDate

#### Summary of Column Name Standardization:

Leading and trailing spaces were removed from all column names in the Sold and List datasets. The updated column lists were reviewed to confirm consistent naming and prevent column-selection errors caused by hidden whitespace.

### 6. Standardize Data Types

#### 6.1 Convert Date Columns

In [11]:
date_columns = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

In [12]:
def convert_date_columns(df, date_columns, dataset_name):
    df = df.copy()

    for col in date_columns:
        if col in df.columns:
            original_non_null = df[col].notna().sum()

            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

            converted_non_null = df[col].notna().sum()
            invalid_count = original_non_null - converted_non_null

            print(
                f"{dataset_name} | {col}: "
                f"{invalid_count:,} values became NaT"
            )
        else:
            print(f"{dataset_name} | {col}: column not found")

    return df

In [13]:
sold_clean = convert_date_columns(
    sold_clean,
    date_columns,
    "Sold"
)

list_clean = convert_date_columns(
    list_clean,
    date_columns,
    "List"
)

Sold | CloseDate: 0 values became NaT
Sold | PurchaseContractDate: 0 values became NaT
Sold | ListingContractDate: 0 values became NaT
Sold | ContractStatusChangeDate: 0 values became NaT
List | CloseDate: 0 values became NaT
List | PurchaseContractDate: 0 values became NaT
List | ListingContractDate: 0 values became NaT
List | ContractStatusChangeDate: 0 values became NaT


In [14]:
# check the date cols' data type is datetime
for col in date_columns:
    if col in sold_clean.columns:
        print(col, sold_clean[col].dtype)

CloseDate datetime64[ns]
PurchaseContractDate datetime64[ns]
ListingContractDate datetime64[ns]
ContractStatusChangeDate datetime64[ns]


##### Check Datetime Formats (Make Sure They Have Not Changed)

In [15]:
def inspect_datetime_values(df, date_columns, dataset_name):
    print(f"\n{dataset_name} datetime review:")

    for col in date_columns:
        if col not in df.columns:
            print(f"\n{col}: column not found")
            continue

        print(f"\nColumn: {col}")
        print(f"Dtype: {df[col].dtype}")
        print("Sample values:")

        print(
            df[col]
            .dropna()
            .astype(str)
            .head(2)
            .to_string(index=False)
        )

In [16]:
inspect_datetime_values(
    sold_clean,
    date_columns,
    "Sold"
)

inspect_datetime_values(
    list_clean,
    date_columns,
    "List"
)


Sold datetime review:

Column: CloseDate
Dtype: datetime64[ns]
Sample values:
2024-01-26
2024-01-05

Column: PurchaseContractDate
Dtype: datetime64[ns]
Sample values:
2023-11-22
2021-06-30

Column: ListingContractDate
Dtype: datetime64[ns]
Sample values:
2021-10-06
2021-03-08

Column: ContractStatusChangeDate
Dtype: datetime64[ns]
Sample values:
2024-01-26
2024-01-05

List datetime review:

Column: CloseDate
Dtype: datetime64[ns]
Sample values:
2024-05-01
2024-03-15

Column: PurchaseContractDate
Dtype: datetime64[ns]
Sample values:
2024-05-07
2024-05-08

Column: ListingContractDate
Dtype: datetime64[ns]
Sample values:
2024-01-01
2024-01-24

Column: ContractStatusChangeDate
Dtype: datetime64[ns]
Sample values:
2024-05-07
2024-01-24


##### Summary of Datetime Conversion:

The four date columns in both the Sold and List datasets were successfully converted to datetime64[ns]. No additional values became NaT during conversion, indicating that all existing non-missing dates were parsed successfully. Sample values confirmed a consistent YYYY-MM-DD format.

#### 6.2 Convert boolean columns

In [17]:
boolean_columns = [
    "AttachedGarageYN",
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "FireplaceYN",
    "WaterfrontYN",
    "BasementYN"
]

In [18]:
print("Sold boolean column dtypes:")
print(
    sold_clean[
        [col for col in boolean_columns if col in sold_clean.columns]
    ].dtypes
)

print("\nList boolean column dtypes:")
print(
    list_clean[
        [col for col in boolean_columns if col in list_clean.columns]
    ].dtypes
)

Sold boolean column dtypes:
AttachedGarageYN     object
ViewYN               object
PoolPrivateYN        object
NewConstructionYN    object
FireplaceYN          object
WaterfrontYN         object
BasementYN           object
dtype: object

List boolean column dtypes:
AttachedGarageYN     object
NewConstructionYN    object
FireplaceYN          object
dtype: object


In [19]:
# To verify they are boolean
for col in boolean_columns:
    if col in sold_clean.columns:
        print("\n" + "-" * 60)
        print(f"Sold | {col}")
        print("Column dtype:", sold_clean[col].dtype)
        print(sold_clean[col].value_counts(dropna=False))

for col in boolean_columns:
    if col in list_clean.columns:
        print("\n" + "-" * 60)
        print(f"List | {col}")
        print("Column dtype:", list_clean[col].dtype)
        print(list_clean[col].value_counts(dropna=False))        

# It should have these results
# <class 'bool'>
# <class 'pandas._libs.missing.NAType'>


------------------------------------------------------------
Sold | AttachedGarageYN
Column dtype: object
AttachedGarageYN
True     301331
False     78232
NaN       68470
Name: count, dtype: int64

------------------------------------------------------------
Sold | ViewYN
Column dtype: object
ViewYN
True     249484
False    160119
NaN       38430
Name: count, dtype: int64

------------------------------------------------------------
Sold | PoolPrivateYN
Column dtype: object
PoolPrivateYN
False    353821
True      55957
NaN       38255
Name: count, dtype: int64

------------------------------------------------------------
Sold | NewConstructionYN
Column dtype: object
NewConstructionYN
False    397769
NaN       33419
True      16845
Name: count, dtype: int64

------------------------------------------------------------
Sold | FireplaceYN
Column dtype: object
FireplaceYN
True     295966
False    151708
NaN         359
Name: count, dtype: int64

-------------------------------------------

In [20]:
# Convert them into boolean
boolean_mapping = {
    True: True,
    False: False,
    "True": True,
    "False": False,
    "Y": True,
    "N": False
}

for col in boolean_columns:
    if col in sold_clean.columns:
        sold_clean[col] = (
            sold_clean[col]
            .map(boolean_mapping)
            .astype("boolean")
        )

    if col in list_clean.columns:
        list_clean[col] = (
            list_clean[col]
            .map(boolean_mapping)
            .astype("boolean")
        )

In [21]:
# Verify conversion (The data type shouldn't be object )
print("Sold Boolean columns:")
print(
    sold_clean.select_dtypes(include=["boolean", "bool"]).dtypes
)

print("\nList Boolean columns:")
print(
    list_clean.select_dtypes(include=["boolean", "bool"]).dtypes
)

Sold Boolean columns:
ViewYN               boolean
WaterfrontYN         boolean
BasementYN           boolean
PoolPrivateYN        boolean
AttachedGarageYN     boolean
FireplaceYN          boolean
NewConstructionYN    boolean
dtype: object

List Boolean columns:
AttachedGarageYN     boolean
FireplaceYN          boolean
NewConstructionYN    boolean
dtype: object


##### Summary of Boolean Column Conversion:

Boolean indicator columns were reviewed and converted from object to pandas boolean data types while preserving missing values. Seven columns were converted in the Sold dataset: ViewYN, WaterfrontYN, BasementYN, PoolPrivateYN, AttachedGarageYN, FireplaceYN, and NewConstructionYN. Three columns were converted in the List dataset: AttachedGarageYN, FireplaceYN, and NewConstructionYN. No records were removed during this step.

#### 6.3 Convert Numeric Columns

In [22]:
# define the numeric columns
numeric_columns = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "LotSizeAcres",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "BathroomsFull",
    "BathroomsHalf",
    "DaysOnMarket",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "TaxAnnualAmount",
    "ParkingTotal",
    "FireplacesTotal",
    "BuildingAreaTotal",
    "AboveGradeFinishedArea"
]

In [23]:
# create a function
def convert_numeric_columns(df, numeric_columns, dataset_name):
    df = df.copy()
    conversion_summary = []

    for col in numeric_columns:
        if col not in df.columns:
            continue

        original_non_null = df[col].notna().sum()
        original_dtype = str(df[col].dtype)

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        converted_non_null = df[col].notna().sum()

        conversion_summary.append({
            "dataset": dataset_name,
            "column": col,
            "original_dtype": original_dtype,
            "new_dtype": str(df[col].dtype),
            "values_converted_to_nan":
                original_non_null - converted_non_null
        })

    return df, pd.DataFrame(conversion_summary)

In [24]:
sold_clean, sold_numeric_conversion = convert_numeric_columns(
    sold_clean,
    numeric_columns,
    "Sold"
)

list_clean, list_numeric_conversion = convert_numeric_columns(
    list_clean,
    numeric_columns,
    "List"
)

In [25]:
print(sold_numeric_conversion.to_string(index=False))
print()
print(list_numeric_conversion.to_string(index=False))

dataset                 column original_dtype new_dtype  values_converted_to_nan
   Sold             ClosePrice        float64   float64                        0
   Sold              ListPrice        float64   float64                        0
   Sold      OriginalListPrice        float64   float64                        0
   Sold             LivingArea        float64   float64                        0
   Sold           LotSizeAcres        float64   float64                        0
   Sold          BedroomsTotal        float64   float64                        0
   Sold  BathroomsTotalInteger        float64   float64                        0
   Sold           DaysOnMarket          int64     int64                        0
   Sold              YearBuilt        float64   float64                        0
   Sold               Latitude        float64   float64                        0
   Sold              Longitude        float64   float64                        0
   Sold        TaxAnnualAmou

##### Summary of Numeric Column Conversion:

Selected price, property-size, bedroom, bathroom, market-time, location, tax, and building-related fields were converted or confirmed as numeric in both the Sold and List datasets. Most columns remained float64, while DaysOnMarket remained int64. No existing non-missing values were converted to NaN, indicating that the numeric conversion did not cause data loss.

#### 6.4 Check Object columns

In [26]:
# Review object numeric columns
def review_object_numeric_columns(df, dataset_name):
    review_summary = []

    object_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for col in object_columns:
        original_non_null = df[col].notna().sum()

        converted = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        numeric_non_null = converted.notna().sum()

        numeric_rate = (
            numeric_non_null / original_non_null
            if original_non_null > 0
            else 0
        )

        review_summary.append({
            "dataset": dataset_name,
            "column": col,
            "current_dtype": str(df[col].dtype),
            "non_null_count": original_non_null,
            "numeric_value_count": numeric_non_null,
            "numeric_rate": numeric_rate,
            "sample_values": ", ".join(
                df[col]
                .dropna()
                .astype(str)
                .unique()[:5]
            )
        })

    return (
        pd.DataFrame(review_summary)
        .sort_values("numeric_rate", ascending=False)
        .reset_index(drop=True)
    )

In [27]:
sold_object_review = review_object_numeric_columns(
    sold_clean,
    "Sold"
)

list_object_review = review_object_numeric_columns(
    list_clean,
    "List"
)

display(sold_object_review.head(5))
display(list_object_review.head(5))

,dataset,column,current_dtype,non_null_count,numeric_value_count,numeric_rate,sample_values
0,Sold,lonfilled,object,63884,63884,1.000000,"False, True"
1,Sold,latfilled,object,63884,63884,1.000000,"False, True"
2,Sold,PostalCode,object,448029,445701,0.994804,"94401, 91950, 92262, 91356, 93401"
3,Sold,ListingId,object,448033,82564,0.184281,"ML81865679, 210014555, 210008330, ML81975626, ..."
4,Sold,LotSizeDimensions,object,21822,2923,0.133947,"47x282, 89x193, 30x75, 84x119, 60x150"


,dataset,column,current_dtype,non_null_count,numeric_value_count,numeric_rate,sample_values
0,List,PostalCode,object,615718,612521,0.994808,"90067, 92677, 91108, 91765, 92662"
1,List,ListingId,object,615739,124929,0.202893,"24389291, AR24091628, P1-17547, OC24091496, NP..."
2,List,LotSizeDimensions,object,32271,3536,0.109572,"21344 SF, 97x112x110x34x23, 3372, 6447, 50x118"
3,List,BuyerAgentMlsId,object,162916,7580,0.046527,"OCDOYLAND, ATLEWISL, PKRAMROC, OCDUNNJAM, CCBE..."
4,List,BuilderName,object,28315,22,0.000777,"Taylor Morrison, Ross Cortese, Pacesetter, Cen..."


In [28]:
display(
    sold_object_review[
        sold_object_review["numeric_rate"] >= 0.95
    ]
)

display(
    list_object_review[
        list_object_review["numeric_rate"] >= 0.95
    ]
)

,dataset,column,current_dtype,non_null_count,numeric_value_count,numeric_rate,sample_values
0,Sold,lonfilled,object,63884,63884,1.000000,"False, True"
1,Sold,latfilled,object,63884,63884,1.000000,"False, True"
2,Sold,PostalCode,object,448029,445701,0.994804,"94401, 91950, 92262, 91356, 93401"


,dataset,column,current_dtype,non_null_count,numeric_value_count,numeric_rate,sample_values
0,List,PostalCode,object,615718,612521,0.994808,"90067, 92677, 91108, 91765, 92662"


##### Convert latfilled and lonfilled columns into boolean data type

In [29]:
boolean_columns = ["latfilled", "lonfilled"]

for col in boolean_columns:
    if col in sold_clean.columns:
        sold_clean[col] = (
            sold_clean[col]
            .map({
                True: True,
                False: False,
                "True": True,
                "False": False
            })
            .astype("boolean")
        )

#### Summary of object columns
Object-type columns were reviewed for numeric content. latfilled and lonfilled were identified as Boolean indicators and converted to Boolean data types. PostalCode was retained as text because it is a geographic identifier rather than a numeric measurement. No displayed columns required numeric conversion.

### 7. Remove Duplicate Columns

#### Find out the duplicate columns

In [30]:
sold_dot_one_columns = [
    col for col in sold_clean.columns
    if col.endswith(".1")
]

list_dot_one_columns = [
    col for col in list_clean.columns
    if col.endswith(".1")
]

print("Sold .1 columns:")
print(sold_dot_one_columns)

print("\nList .1 columns:")
print(list_dot_one_columns)

Sold .1 columns:
[]

List .1 columns:
['PropertyType.1', 'ListAgentFirstName.1', 'DaysOnMarket.1', 'LivingArea.1', 'Longitude.1', 'Latitude.1', 'ListPrice.1', 'ListAgentLastName.1', 'CloseDate.1', 'BuyerOfficeName.1', 'UnparsedAddress.1']


#### Compare original columns and .1 columns

In [31]:
def compare_dot_one_columns(df):
    comparison_results = []

    dot_one_columns = [
        col for col in df.columns
        if col.endswith(".1")
    ]

    for duplicate_col in dot_one_columns:
        original_col = duplicate_col[:-2]

        if original_col not in df.columns:
            comparison_results.append({
                "original_column": original_col,
                "duplicate_column": duplicate_col,
                "original_exists": False,
                "same_values": False,
                "different_rows": None
            })
            continue

        same_mask = (
            df[original_col].eq(df[duplicate_col]) |
            (
                df[original_col].isna() &
                df[duplicate_col].isna()
            )
        )

        comparison_results.append({
            "original_column": original_col,
            "duplicate_column": duplicate_col,
            "original_exists": True,
            "same_values": bool(same_mask.all()),
            "different_rows": int((~same_mask).sum())
        })

    return pd.DataFrame(comparison_results)

In [32]:
sold_duplicate_column_check = compare_dot_one_columns(sold_clean)
list_duplicate_column_check = compare_dot_one_columns(list_clean)

print("Sold duplicate-column comparison:")
print(sold_duplicate_column_check.to_string(index=False))

print("\nList duplicate-column comparison:")
print(list_duplicate_column_check.to_string(index=False))

Sold duplicate-column comparison:
Empty DataFrame
Columns: []
Index: []

List duplicate-column comparison:
   original_column     duplicate_column  original_exists  same_values  different_rows
      PropertyType       PropertyType.1             True         True               0
ListAgentFirstName ListAgentFirstName.1             True         True               0
      DaysOnMarket       DaysOnMarket.1             True         True               0
        LivingArea         LivingArea.1             True         True               0
         Longitude          Longitude.1             True         True               0
          Latitude           Latitude.1             True         True               0
         ListPrice          ListPrice.1             True         True               0
 ListAgentLastName  ListAgentLastName.1             True         True               0
         CloseDate          CloseDate.1             True         True               0
   BuyerOfficeName    BuyerOffice

#### Only delete the duplicate columns have the same values

In [33]:
list_duplicate_columns_to_drop = (
    list_duplicate_column_check
    .loc[
        list_duplicate_column_check["same_values"],
        "duplicate_column"
    ]
    .tolist()
)

print("List columns to drop:")
print(list_duplicate_columns_to_drop)

List columns to drop:
['PropertyType.1', 'ListAgentFirstName.1', 'DaysOnMarket.1', 'LivingArea.1', 'Longitude.1', 'Latitude.1', 'ListPrice.1', 'ListAgentLastName.1', 'CloseDate.1', 'BuyerOfficeName.1', 'UnparsedAddress.1']


#### Delete duplicate columns and document it into cleaning_log

In [34]:
rows_before, cols_before = list_clean.shape

list_clean = list_clean.drop(
    columns=list_duplicate_columns_to_drop
)

rows_after, cols_after = list_clean.shape

add_cleaning_log(
    dataset_name="List",
    step="Remove duplicate .1 columns",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes=f"Dropped: {list_duplicate_columns_to_drop}"
)

#### Summary of Duplicate Column Review and Removal:

No .1 duplicate columns were found in the Sold dataset. In the List dataset, the .1 columns—including PropertyType.1, ListAgentFirstName.1, DaysOnMarket.1, LivingArea.1, Longitude.1, Latitude.1, ListPrice.1, ListAgentLastName.1, CloseDate.1, BuyerOfficeName.1, and UnparsedAddress.1—were compared with their corresponding original columns. All pairs contained identical values with zero differing records. Therefore, these confirmed duplicate columns were removed without deleting any rows or losing information, and the changes were recorded in the cleaning log.

### 8. Missing Value Analysis

#### Missing Value Summary

In [35]:
# create missing value summary function
def create_missing_summary(df):
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null_count": df.notna().sum().values,
        "missing_count": df.isna().sum().values,
        "missing_percent": (
            df.isna().mean() * 100
        ).round(2).values,
        "unique_count": df.nunique(dropna=True).values
    })

    return summary.sort_values(
        by="missing_percent",
        ascending=False
    ).reset_index(drop=True)

In [36]:
sold_missing_summary = create_missing_summary(sold_clean)
list_missing_summary = create_missing_summary(list_clean)

print("Sold missing summary:")
print(sold_missing_summary.to_string(index=False))

print("\nList missing summary:")
print(list_missing_summary.to_string(index=False))

Sold missing summary:
                      column          dtype  non_null_count  missing_count  missing_percent  unique_count
             TaxAnnualAmount        float64               0         448033           100.00             0
MiddleOrJuniorSchoolDistrict        float64               0         448033           100.00             0
                BusinessType        float64               0         448033           100.00             0
    ElementarySchoolDistrict        float64               0         448033           100.00             0
                     TaxYear        float64               0         448033           100.00             0
             FireplacesTotal        float64               0         448033           100.00             0
      AboveGradeFinishedArea        float64               0         448033           100.00             0
               CoveredSpaces        float64               0         448033           100.00             0
                Waterfro

#### Review columns with over 90% missing rate

In [37]:
sold_over_90_missing = sold_missing_summary[
    sold_missing_summary["missing_percent"] > 90
]

list_over_90_missing = list_missing_summary[
    list_missing_summary["missing_percent"] > 90
]

print("Sold columns over 90% missing:")
print(sold_over_90_missing.to_string(index=False))

print("\nList columns over 90% missing:")
print(list_over_90_missing.to_string(index=False))

Sold columns over 90% missing:
                      column   dtype  non_null_count  missing_count  missing_percent  unique_count
             TaxAnnualAmount float64               0         448033           100.00             0
MiddleOrJuniorSchoolDistrict float64               0         448033           100.00             0
                BusinessType float64               0         448033           100.00             0
    ElementarySchoolDistrict float64               0         448033           100.00             0
                     TaxYear float64               0         448033           100.00             0
             FireplacesTotal float64               0         448033           100.00             0
      AboveGradeFinishedArea float64               0         448033           100.00             0
               CoveredSpaces float64               0         448033           100.00             0
                WaterfrontYN boolean             287         447746           

#### Review columns with 50% to 90% missing rate

In [38]:
sold_50_to_90_missing = sold_missing_summary[
    sold_missing_summary["missing_percent"].between(
        50,
        90,
        inclusive="both"
    )
]

list_50_to_90_missing = list_missing_summary[
    list_missing_summary["missing_percent"].between(
        50,
        90,
        inclusive="both"
    )
]

print("Sold columns 50% to 90% missing:")
print(sold_50_to_90_missing.to_string(index=False))

print("\nList columns 50% to 90% missing:")
print(list_50_to_90_missing.to_string(index=False))

Sold columns 50% to 90% missing:
                     column   dtype  non_null_count  missing_count  missing_percent  unique_count
    BuyerAgencyCompensation float64           46125         401908            89.70           174
BuyerAgencyCompensationType  object           46136         401897            89.70             3
           ElementarySchool  object           59523         388510            86.71          1409
       MiddleOrJuniorSchool  object           59945         388088            86.62           662
                  lonfilled boolean           63884         384149            85.74             2
                  latfilled boolean           63884         384149            85.74             2
                 HighSchool  object           77857         370176            82.62           521
      OriginatingSystemName  object           89682         358351            79.98             1
   OriginatingSystemSubName  object           89682         358351            79.98  

#### Drop over 90% missing rate columns

In [39]:
# make over 90% missing rate columns into a list
sold_cols_to_drop = sold_missing_summary.loc[
    sold_missing_summary["missing_percent"] > 90,
    "column"
].tolist()

list_cols_to_drop = list_missing_summary.loc[
    list_missing_summary["missing_percent"] > 90,
    "column"
].tolist()

print("Sold columns to drop:")
print(sold_cols_to_drop)

print("\nList columns to drop:")
print(list_cols_to_drop)

Sold columns to drop:
['TaxAnnualAmount', 'MiddleOrJuniorSchoolDistrict', 'BusinessType', 'ElementarySchoolDistrict', 'TaxYear', 'FireplacesTotal', 'AboveGradeFinishedArea', 'CoveredSpaces', 'WaterfrontYN', 'BelowGradeFinishedArea', 'BasementYN', 'LotSizeDimensions', 'BuilderName', 'BuildingAreaTotal', 'CoBuyerAgentFirstName']

List columns to drop:
['TaxYear', 'BusinessType', 'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict', 'TaxAnnualAmount', 'AboveGradeFinishedArea', 'FireplacesTotal', 'CoveredSpaces', 'BelowGradeFinishedArea', 'CoBuyerAgentFirstName', 'BuilderName', 'LotSizeDimensions', 'BuildingAreaTotal']


In [40]:
# drop over 90% missig rate columns
sold_clean = sold_clean.drop(
    columns=sold_cols_to_drop
)

list_clean = list_clean.drop(
    columns=list_cols_to_drop
)

In [41]:
# Verify the result
print("Sold shape after dropping columns:", sold_clean.shape)
print("List shape after dropping columns:", list_clean.shape)

Sold shape after dropping columns: (448033, 74)
List shape after dropping columns: (615739, 65)


In [42]:
# Verify if over 90% columns has been drop
print(
    "Sold dropped columns still present:",
    [col for col in sold_cols_to_drop if col in sold_clean.columns]
)

print(
    "List dropped columns still present:",
    [col for col in list_cols_to_drop if col in list_clean.columns]
)

Sold dropped columns still present: []
List dropped columns still present: []


#### Summary of Missing Value Analysis:

Missing values were reviewed at the column level for both datasets. Columns with more than 90% missing values were removed because they contained limited usable information for analysis. This reduced the Sold dataset from 89 to 74 columns and the List dataset from 89 to 65 columns, while keeping all rows unchanged. A final check confirmed that none of the selected high-missing columns remained in the cleaned datasets.

### 9. Check Sold dataset 2 missing ClosePrice rows

In [43]:
sold_missing_close_price = sold_clean[
    sold_clean["ClosePrice"].isna()
].copy()

print("Sold rows with missing ClosePrice:")
print(sold_missing_close_price.shape)

display(sold_missing_close_price)

Sold rows with missing ClosePrice:
(2, 74)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,OriginatingSystemSubName,source_file,year_month,data_type,file_version,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled,rate_30yr_fixed
92215,MariposaCounty,MariposaCounty,NaN,True,True,575000.0,1045745288,shelby.twissrealty@gmail.com,2024-06-06,NaN,...,NaN,CRMLSSold202406_filled.csv,2024-06,sold,filled,NaN,NaN,True,True,6.9175
182391,MercedCounty,MercedCounty,NaN,True,True,2500.0,1086646611,trustedrealtors4ucarla@gmail.com,2024-12-13,NaN,...,NaN,CRMLSSold202412.csv,2024-12,sold,original,NaN,NaN,<NA>,<NA>,6.7150


In [44]:
# Check the important point
columns_to_review = [
    "ListingKey",
    "ListingId",
    "ClosePrice",
    "ListPrice",
    "CloseDate",
    "PropertyType",
    "PropertySubType",
    "MlsStatus"
]

existing_columns = [
    col for col in columns_to_review
    if col in sold_missing_close_price.columns
]

#display(
#    sold_missing_close_price[existing_columns]
#)

#### Summary of Missing ClosePrice Review:

Two Sold records were identified with missing `ClosePrice` values. These records were reviewed separately because `ClosePrice` is required for sales-price analysis. The records were retained for further review rather than automatically removed.

### 10. Geographic Data Checks

#### Set up California latitude and longitude boundary

In [45]:
CA_LAT_MIN = 32
CA_LAT_MAX = 42

CA_LON_MIN = -125
CA_LON_MAX = -114

In [ ]:
# create a function for geographic flag
def add_geographic_flags(df):
    df = df.copy()

    # 1. Latitude or Longitude is missing
    df["missing_coordinate_flag"] = (
        df["Latitude"].isna()
        | df["Longitude"].isna()
    )

    # 2. Latitude equals zero
    df["lat_zero_flag"] = (
        df["Latitude"].notna()
        & df["Latitude"].eq(0)
    )

    # 3. Longitude equals zero
    df["lon_zero_flag"] = (
        df["Longitude"].notna()
        & df["Longitude"].eq(0)
    )

    # Only evaluate out-of-state when coordinates are present and non-zero
    valid_nonzero_coordinates = (
        df["Latitude"].notna()
        & df["Longitude"].notna()
        & df["Latitude"].ne(0)
        & df["Longitude"].ne(0)
    )

    # 4. Coordinates outside the expected California range
    df["out_of_state_flag"] = (
        valid_nonzero_coordinates
        & (
            (df["Longitude"] > 0)
            | (df["Latitude"] < CA_LAT_MIN)
            | (df["Latitude"] > CA_LAT_MAX)
            | (df["Longitude"] < CA_LON_MIN)
            | (df["Longitude"] > CA_LON_MAX)
        )
    )

    return df

In [47]:
sold_clean = add_geographic_flags(sold_clean)
list_clean = add_geographic_flags(list_clean)

#### Check geographic flags

In [48]:
geographic_flags = [
    "missing_coordinate_flag",
    "lat_zero_flag",
    "lon_zero_flag",
    "out_of_state_flag"
]

print("Sold geographic flags:")
print(
    sold_clean[geographic_flags]
    .sum()
    .to_string()
)

print("\nList geographic flags:")
print(
    list_clean[geographic_flags]
    .sum()
    .to_string()
)

Sold geographic flags:
missing_coordinate_flag    16221
lat_zero_flag                 37
lon_zero_flag                 37
out_of_state_flag             99

List geographic flags:
missing_coordinate_flag    81000
lat_zero_flag                 75
lon_zero_flag                 75
out_of_state_flag            323


#### Summary of Geographic Data Quality Flags:

Missing coordinates were the primary geographic data issue, affecting 16,221 Sold records and 81,000 List records. Zero-coordinate and out-of-state values were uncommon in both datasets. Missing, zero, and invalid coordinates were flagged separately for clearer data-quality tracking, and no records were removed during this step.

### 11. Invalid Numeric Value Checks

#### Invalid numeric-values flag

In [49]:
def add_invalid_numeric_flags(df, dataset_type):
    df = df.copy()

    if dataset_type == "sold" and "ClosePrice" in df.columns:
        df["invalid_close_price_flag"] = (
            df["ClosePrice"].notna() &
            df["ClosePrice"].le(0)
        )

    if "ListPrice" in df.columns:
        df["invalid_list_price_flag"] = (
            df["ListPrice"].notna() &
            df["ListPrice"].le(0)
        )

    if "LivingArea" in df.columns:
        df["invalid_living_area_flag"] = (
            df["LivingArea"].notna() &
            df["LivingArea"].le(0)
        )

    if "DaysOnMarket" in df.columns:
        df["negative_days_on_market_flag"] = (
            df["DaysOnMarket"].notna() &
            df["DaysOnMarket"].lt(0)
        )

    if "BedroomsTotal" in df.columns:
        df["negative_bedrooms_flag"] = (
            df["BedroomsTotal"].notna() &
            df["BedroomsTotal"].lt(0)
        )

    if "BathroomsTotalInteger" in df.columns:
        df["negative_bathrooms_flag"] = (
            df["BathroomsTotalInteger"].notna() &
            df["BathroomsTotalInteger"].lt(0)
        )

    return df

In [50]:
sold_clean = add_invalid_numeric_flags(
    sold_clean,
    dataset_type="sold"
)

list_clean = add_invalid_numeric_flags(
    list_clean,
    dataset_type="list"
)

#### Check invalid-value counts

In [51]:
sold_invalid_flag_columns = [
    col for col in sold_clean.columns
    if col.startswith("invalid_") or col.startswith("negative_")
]

list_invalid_flag_columns = [
    col for col in list_clean.columns
    if col.startswith("invalid_") or col.startswith("negative_")
]

print("Sold invalid numeric flags:")
print(
    sold_clean[sold_invalid_flag_columns]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

print("\nList invalid numeric flags:")
print(
    list_clean[list_invalid_flag_columns]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

Sold invalid numeric flags:
invalid_living_area_flag        165
negative_days_on_market_flag     50
invalid_close_price_flag          1
invalid_list_price_flag           0
negative_bedrooms_flag            0
negative_bathrooms_flag           0

List invalid numeric flags:
invalid_living_area_flag        393
negative_days_on_market_flag     30
invalid_list_price_flag           0
negative_bedrooms_flag            0
negative_bathrooms_flag           0


#### Summary of Invalid Numeric Value Check:

Invalid numeric values were reviewed and flagged in both datasets. Non-positive living area was the most common issue, affecting 165 Sold records and 393 List records. Negative days on market were found in 50 Sold records and 30 List records, while only one Sold record had an invalid close price. No invalid list prices or negative bedroom and bathroom values were identified. All affected records were retained and flagged for future review.

### 12. Date consistency flags

In [52]:
# create a function
def add_date_consistency_flags(df, label):
    df = df.copy()

    # Listing date occurs after the closing date
    df["listing_after_close_flag"] = (
        df["ListingContractDate"].notna()
        & df["CloseDate"].notna()
        & (
            df["ListingContractDate"]
            > df["CloseDate"]
        )
    )

    # Purchase contract date occurs after the closing date
    df["purchase_after_close_flag"] = (
        df["PurchaseContractDate"].notna()
        & df["CloseDate"].notna()
        & (
            df["PurchaseContractDate"]
            > df["CloseDate"]
        )
    )

    # Purchase contract date occurs before the listing date
    purchase_before_listing = (
        df["ListingContractDate"].notna()
        & df["PurchaseContractDate"].notna()
        & (
            df["PurchaseContractDate"]
            < df["ListingContractDate"]
        )
    )

    # Overall timeline inconsistency flag
    df["negative_timeline_flag"] = (
        df["listing_after_close_flag"]
        | df["purchase_after_close_flag"]
        | purchase_before_listing
    )

    print(
        f"- [{label}] listing_after_close_flag True: "
        f"{df['listing_after_close_flag'].sum():,}"
    )
    print(
        f"- [{label}] purchase_after_close_flag True: "
        f"{df['purchase_after_close_flag'].sum():,}"
    )
    print(
        f"- [{label}] purchase_before_listing condition True: "
        f"{purchase_before_listing.sum():,}"
    )
    print(
        f"- [{label}] negative_timeline_flag True: "
        f"{df['negative_timeline_flag'].sum():,}"
    )

    return df

In [53]:
sold_clean = add_date_consistency_flags(
    sold_clean,
    "Sold"
)

list_clean = add_date_consistency_flags(
    list_clean,
    "List"
)

- [Sold] listing_after_close_flag True: 68
- [Sold] purchase_after_close_flag True: 241
- [Sold] purchase_before_listing condition True: 290
- [Sold] negative_timeline_flag True: 531
- [List] listing_after_close_flag True: 84
- [List] purchase_after_close_flag True: 268
- [List] purchase_before_listing condition True: 301
- [List] negative_timeline_flag True: 568


#### Summary of Date Consistency Flags:

Date consistency checks identified 531 Sold records and 568 List records with at least one timeline issue. These issues included listing dates after closing, purchase contract dates after closing, or purchase contract dates before listing. Some records met more than one condition, so individual flag counts may overlap. All date-related flags were retained, and no records were removed solely because of these flags.

### 13. Duplicate-row review

In [55]:
print(
    "Sold fully duplicated rows:",
    sold_clean.duplicated().sum()
)

print(
    "List fully duplicated rows:",
    list_clean.duplicated().sum()
)

Sold fully duplicated rows: 48
List fully duplicated rows: 0


In [56]:
# create a function to check ListingKey
def review_listing_key_duplicates(df, dataset_name):
    if "ListingKey" not in df.columns:
        print(dataset_name, "does not contain ListingKey")
        return pd.DataFrame()

    duplicate_mask = df.duplicated(
        subset=["ListingKey"],
        keep=False
    )

    duplicate_records = (
        df.loc[duplicate_mask]
        .sort_values("ListingKey")
    )

    print(
        dataset_name,
        "rows with duplicated ListingKey:",
        len(duplicate_records)
    )

    print(
        dataset_name,
        "duplicated ListingKey count:",
        duplicate_records["ListingKey"].nunique()
    )

    return duplicate_records

In [57]:
sold_listing_key_duplicates = review_listing_key_duplicates(
    sold_clean,
    "Sold"
)

list_listing_key_duplicates = review_listing_key_duplicates(
    list_clean,
    "List"
)

Sold rows with duplicated ListingKey: 744
Sold duplicated ListingKey count: 370
List rows with duplicated ListingKey: 376
List duplicated ListingKey count: 187


We cannot remove by the same ListingKey because the data might come from different year and month dataset, but they have the same ListingKey.

In [58]:
duplicate_review_columns = [
    "ListingKey",
    "ListingId",
    "year_month",
    "source_file",
    "file_version",
    "MlsStatus",
    "ListPrice",
    "ClosePrice",
    "CloseDate"
]

existing_columns = [
    col for col in duplicate_review_columns
    if col in sold_listing_key_duplicates.columns
]

display(
    sold_listing_key_duplicates[existing_columns].head(10)
)

,ListingKey,ListingId,year_month,source_file,file_version,MlsStatus,ListPrice,ClosePrice,CloseDate
92419,1031992525,SW22259219,2024-06,CRMLSSold202406_filled.csv,filled,Closed,415740.0,415740.0,2024-06-21
157865,1031992525,SW22259219,2024-10,CRMLSSold202410.csv,original,Closed,459900.0,459900.0,2024-10-18
141580,1037464932,PW23082927,2024-09,CRMLSSold202409.csv,original,Closed,675000.0,665000.0,2024-09-06
75731,1037464932,PW23082927,2024-05,CRMLSSold202405_filled.csv,filled,Closed,675000.0,650000.0,2024-05-25
92367,1038376137,OC23108406,2024-06,CRMLSSold202406_filled.csv,filled,Closed,1732000.0,1695000.0,2024-06-10
75689,1038376137,OC23108406,2024-05,CRMLSSold202405_filled.csv,filled,Closed,1732000.0,1695000.0,2024-05-27
92081,1046689006,PI23203651,2024-06,CRMLSSold202406_filled.csv,filled,Closed,825000.0,825000.0,2024-06-30
157786,1046689006,PI23203651,2024-10,CRMLSSold202410.csv,original,Closed,825000.0,825000.0,2024-10-29
91945,1048638478,SR23217206,2024-06,CRMLSSold202406_filled.csv,filled,Closed,6700000.0,6600000.0,2024-06-25
126850,1048638478,SR23217206,2024-08,CRMLSSold202408.csv,original,Closed,6700000.0,6550000.0,2024-08-21


#### Remove fully duplicate rows

#### Sold

In [59]:
rows_before, cols_before = sold_clean.shape

sold_clean = sold_clean.drop_duplicates().copy()

rows_after, cols_after = sold_clean.shape

add_cleaning_log(
    dataset_name="Sold",
    step="Remove fully duplicated rows",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes="Removed rows identical across all columns"
)

#### List

In [60]:
rows_before, cols_before = list_clean.shape

list_clean = list_clean.drop_duplicates().copy()

rows_after, cols_after = list_clean.shape

add_cleaning_log(
    dataset_name="List",
    step="Remove fully duplicated rows",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes="Removed rows identical across all columns"
)

#### Summary of Duplicate Row Review:

The Sold dataset contained 48 fully duplicated rows, while no fully duplicated rows were found in the List dataset. Duplicate ListingKey values were also identified in both datasets, but these records were retained because the same property may appear in different months or source files with updated prices, statuses, or closing dates. Therefore, only rows identical across all columns were removed, and the changes were documented in the cleaning log.

### 14. Remove invalid values

#### 14.1 Sold dataset removal standard

In [61]:
# Sold removal standard
sold_removal_flags = [
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "negative_days_on_market_flag",
    "negative_bedrooms_flag",
    "negative_bathrooms_flag"
]

sold_removal_flags = [
    col for col in sold_removal_flags
    if col in sold_clean.columns
]

sold_remove_mask = sold_clean[
    sold_removal_flags
].any(axis=1)

print("Sold rows proposed for removal:")
print(sold_remove_mask.sum())

Sold rows proposed for removal:
216


In [62]:
# Check these flag's rows
sold_invalid_records = sold_clean.loc[
    sold_remove_mask
].copy()

#display(sold_invalid_records.head(30))

In [63]:
# remove Sold invalid values
rows_before, cols_before = sold_clean.shape

sold_clean = sold_clean.loc[
    ~sold_remove_mask
].copy()

rows_after, cols_after = sold_clean.shape

add_cleaning_log(
    dataset_name="Sold",
    step="Remove invalid numeric records",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes=", ".join(sold_removal_flags)
)

#### 14.2 List dataset removal standard

In [64]:
list_removal_flags = [
    "invalid_list_price_flag",
    "invalid_living_area_flag",
    "negative_days_on_market_flag",
    "negative_bedrooms_flag",
    "negative_bathrooms_flag"
]

list_removal_flags = [
    col for col in list_removal_flags
    if col in list_clean.columns
]

list_remove_mask = list_clean[
    list_removal_flags
].any(axis=1)

print("List rows proposed for removal:")
print(list_remove_mask.sum())

List rows proposed for removal:
423


In [65]:
# review removal rows
list_invalid_records = list_clean.loc[
    list_remove_mask
].copy()

#display(list_invalid_records.head(30))

In [66]:
# remove list invalid values
rows_before, cols_before = list_clean.shape

list_clean = list_clean.loc[
    ~list_remove_mask
].copy()

rows_after, cols_after = list_clean.shape

add_cleaning_log(
    dataset_name="List",
    step="Remove invalid numeric records",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes=", ".join(list_removal_flags)
)

#### Summary of Invalid Numeric Record Removal:

Records with invalid core numeric values were removed from the final analysis-ready datasets. A record was removed if it had a non-positive price or living area, negative days on market, or negative bedroom or bathroom values. This resulted in the removal of 216 Sold records and 423 List records. Each record was counted only once even if it met multiple invalid-value conditions, and all removals were documented in the cleaning log. Date and geographic flags were not used as removal criteria.

### 15. Final validation

#### 15.1 Before/After row counts

In [67]:
summary = pd.DataFrame({
    "Dataset": ["Sold", "List"],
    "Rows Before": [
        sold_raw.shape[0],
        list_raw.shape[0]
    ],
    "Rows After": [
        sold_clean.shape[0],
        list_clean.shape[0]
    ],
    "Rows Removed": [
        sold_raw.shape[0]-sold_clean.shape[0],
        list_raw.shape[0]-list_clean.shape[0]
    ]
})

summary

,Dataset,Rows Before,Rows After,Rows Removed
0,Sold,448033,447769,264
1,List,615739,615316,423


In [68]:
print("Final sold shape:", sold_clean.shape)
print("Final list shape:", list_clean.shape)

print("\nSold duplicate rows:")
print(sold_clean.duplicated().sum())

print("\nList duplicate rows:")
print(list_clean.duplicated().sum())

Final sold shape: (447769, 87)
Final list shape: (615316, 77)

Sold duplicate rows:
0

List duplicate rows:
0


#### 15.2 Check numeric dtype

In [69]:
existing_numeric_columns = [
    col for col in numeric_columns
    if col in sold_clean.columns
]

print(
    sold_clean[existing_numeric_columns]
    .dtypes
    .to_string()
)

ClosePrice               float64
ListPrice                float64
OriginalListPrice        float64
LivingArea               float64
LotSizeAcres             float64
BedroomsTotal            float64
BathroomsTotalInteger    float64
DaysOnMarket               int64
YearBuilt                float64
Latitude                 float64
Longitude                float64
ParkingTotal             float64


In [70]:
print(sold_clean.dtypes.value_counts())

print(list_clean.dtypes.value_counts())

object            40
float64           20
bool              13
boolean            7
datetime64[ns]     4
int64              3
Name: count, dtype: int64
object            35
float64           20
bool              12
datetime64[ns]     4
int64              3
boolean            3
Name: count, dtype: int64


#### 15.3 Check date dtype

In [71]:
existing_date_columns = [
    col for col in date_columns
    if col in sold_clean.columns
]

print(
    sold_clean[existing_date_columns]
    .dtypes
    .to_string()
)

CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]


#### 15.4 Create final missing value summaries

In [72]:
sold_missing_summary_after = create_missing_summary(
    sold_clean
)

list_missing_summary_after = create_missing_summary(
    list_clean
)

In [73]:
print("Cleaned Sold missing value summary")
print(sold_missing_summary_after)

print("Cleaned List missing value summary")
print(list_missing_summary_after)

Cleaned Sold missing value summary
                         column    dtype  non_null_count  missing_count  \
0       BuyerAgencyCompensation  float64           46095         401674   
1   BuyerAgencyCompensationType   object           46106         401663   
2              ElementarySchool   object           59512         388257   
3          MiddleOrJuniorSchool   object           59933         387836   
4                     lonfilled  boolean           63860         383909   
..                          ...      ...             ...            ...   
82               CountyOrParish   object          447769              0   
83                    MlsStatus   object          447769              0   
84                    ListingId   object          447769              0   
85                BedroomsTotal  float64          447757             12   
86       negative_timeline_flag     bool          447769              0   

    missing_percent  unique_count  
0             89.71         

#### 15.5 Validate Final Column Counts and Required Flags

In [ ]:
# Check for Sold and List dataset columns

#print("Sold cleaned columns:")

#for col in sold_clean.columns:
#    print(col)

#print("\nList cleaned columns:")

#for col in list_clean.columns:
#    print(col)

In [ ]:
# Check for duplicate column
final_column_summary = pd.DataFrame({
    "dataset": ["Sold", "List"],
    "column_count": [
        sold_clean.shape[1],
        list_clean.shape[1]
    ],
    "duplicate_column_names": [
        sold_clean.columns.duplicated().sum(),
        list_clean.columns.duplicated().sum()
    ]
})

display(final_column_summary)

,dataset,column_count,duplicate_column_names
0,Sold,87,0
1,List,77,0


In [86]:
required_sold_flags = [
    "missing_coordinate_flag",
    "lat_zero_flag",
    "lon_zero_flag",
    "out_of_state_flag",
    "invalid_close_price_flag",
    "invalid_list_price_flag",
    "invalid_living_area_flag",
    "negative_days_on_market_flag",
    "negative_bedrooms_flag",
    "negative_bathrooms_flag",
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag"
]

required_list_flags = [
    "missing_coordinate_flag",
    "lat_zero_flag",
    "lon_zero_flag",
    "out_of_state_flag",
    "invalid_list_price_flag",
    "invalid_living_area_flag",
    "negative_days_on_market_flag",
    "negative_bedrooms_flag",
    "negative_bathrooms_flag",
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag"
]


In [ ]:
# Check for required flags still exist
print(
    "Missing Sold required flags:",
    [col for col in required_sold_flags if col not in sold_clean.columns]
)

print(
    "Missing List required flags:",
    [col for col in required_list_flags if col not in list_clean.columns]
)

Missing Sold required flags: []
Missing List required flags: []


#### 15.6 Date Consistency Flag counts

In [82]:
date_flags = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag"
]

print("Sold Clean date flag counts")
print(
    sold_clean[date_flags]
    .sum()
)
print("\nList Clean date flag counts")
print(
    list_clean[date_flags]
    .sum()
)

Sold Clean date flag counts
listing_after_close_flag      68
purchase_after_close_flag    241
negative_timeline_flag       530
dtype: int64

List Clean date flag counts
listing_after_close_flag      84
purchase_after_close_flag    268
negative_timeline_flag       567
dtype: int64


#### 15.7 Geographic data quality summary

In [76]:
geo_flags = [
    "missing_coordinate_flag",
    "lat_zero_flag",
    "lon_zero_flag",
    "out_of_state_flag"
]

print("Sold Clean geo flag counts")
print(
    sold_clean[geo_flags]
    .sum()
)

print("\nList Clean geo flag counts")
print(
    list_clean[geo_flags]
    .sum()
)

Sold Clean geo flag counts
missing_coordinate_flag    16217
lat_zero_flag                 37
lon_zero_flag                 37
out_of_state_flag             99
dtype: int64

List Clean geo flag counts
missing_coordinate_flag    80978
lat_zero_flag                 75
lon_zero_flag                 75
out_of_state_flag            322
dtype: int64


#### 15.8 Save Cleaning log

In [77]:
cleaning_log_df = pd.DataFrame(cleaning_log)

display(cleaning_log_df)

,dataset,step,rows_before,rows_after,rows_removed,columns_before,columns_after,columns_removed,notes
0,List,Remove duplicate .1 columns,615739,615739,0,89,78,11,"Dropped: ['PropertyType.1', 'ListAgentFirstNam..."
1,Sold,Remove fully duplicated rows,448033,447985,48,87,87,0,Removed rows identical across all columns
2,List,Remove fully duplicated rows,615739,615739,0,77,77,0,Removed rows identical across all columns
3,Sold,Remove invalid numeric records,447985,447769,216,87,87,0,"invalid_close_price_flag, invalid_living_area_..."
4,List,Remove invalid numeric records,615739,615316,423,77,77,0,"invalid_list_price_flag, invalid_living_area_f..."


In [78]:
# Export
cleaning_log_df.to_csv(
    CLEANED_DIR / "cleaning_log.csv",
    index=False
)

#### 15.9 Save Final Summary

In [79]:
sold_missing_summary_after.to_csv(
    CLEANED_DIR / "sold_missing_summary_after_cleaning.csv",
    index=False
)

list_missing_summary_after.to_csv(
    CLEANED_DIR / "list_missing_summary_after_cleaning.csv",
    index=False
)

In [88]:
# Save Final column list
pd.DataFrame({
    "column": sold_clean.columns
}).to_csv(
    CLEANED_DIR / "sold_final_column_list.csv",
    index=False
)

pd.DataFrame({
    "column": list_clean.columns
}).to_csv(
    CLEANED_DIR / "list_final_column_list.csv",
    index=False
)

#### Summary of Final Validation:

The final validation confirmed that the cleaned Sold dataset contains 447,769 rows and 87 columns, while the cleaned List dataset contains 615,316 rows and 77 columns. A total of 264 Sold records and 423 List records were removed during cleaning. No fully duplicated rows remained in either dataset.

The selected numeric fields retained appropriate numeric data types, and the four required date columns remained in `datetime64[ns]` format. Date-consistency and geographic data-quality flags were preserved in both cleaned datasets. Final missing-value summaries and the cleaning log were generated and exported for documentation.

### 16. Save Cleaned_Sold and Cleaned_List datasets

In [80]:
sold_clean.to_csv(
    CLEANED_DIR / "combined_sold_residential_cleaned.csv",
    index=False
)

list_clean.to_csv(
    CLEANED_DIR / "combined_list_residential_cleaned.csv",
    index=False
)

In [81]:
# Make sure the data is successfully exported
print("Saved files:")

for file in CLEANED_DIR.iterdir():
    print(file.name)

Saved files:
list_missing_summary_after_cleaning.csv
combined_list_residential_cleaned.csv
combined_sold_residential_cleaned.csv
cleaning_log.csv
sold_missing_summary_after_cleaning.csv
